In [1]:
import json

import pandas as pd

from flowadj.config import PROCESSED_DIR
from flowadj.data.store import eval_query

pd.set_option("display.width", 200)
pred = pd.read_parquet(PROCESSED_DIR / "detector_predictions.parquet")
lab = eval_query("SELECT row_id, family_corrected AS family, attempted FROM flows_eval")
df = pred.merge(lab, on="row_id")
held = df[df.split.isin(["dev", "test"])]
meta = json.load(open(PROCESSED_DIR / "detector" / "meta.json"))
print(len(df), len(held))
print(df.split.value_counts())

2099976 737212
split
train     879537
buffer    483227
test      480185
dev       257027
Name: count, dtype: int64


In [2]:
def per_family(d):
    d = d.assign(rf_hit=d.rf_class == d.family)
    return d.groupby("family").agg(
        n_flows=("row_id", "size"),
        rf_class_recall=("rf_hit", "mean"),
        rf_alert_rate=("alert_rf", "mean"),
        if_alert_rate=("alert_if", "mean"),
        any_alert_rate=("alert", "mean"),
    ).round(3)

print("HELD-OUT (dev + test)")
per_family(held)

HELD-OUT (dev + test)


,n_flows,rf_class_recall,rf_alert_rate,if_alert_rate,any_alert_rate
family,,,,,
benign,657032,1.000,0.000,0.005,0.005
botnet,1774,0.986,0.986,0.000,0.986
ddos,50020,1.000,1.000,0.000,1.000
dos,14041,0.551,0.574,0.247,0.574
ftp_bruteforce,1699,0.999,1.000,0.000,1.000
other,68,0.015,0.750,0.221,0.809
portscan,10682,0.998,0.999,0.018,0.999
ssh_bruteforce,1138,0.989,0.989,0.000,0.989
web_attack,758,0.979,0.979,0.000,0.979


In [3]:
for name, d in [("train", df[df.split == "train"]), ("dev", df[df.split == "dev"]), ("test", df[df.split == "test"])]:
    print(name, "RF accuracy:", round((d.rf_class == d.family).mean(), 4))
per_family(df[df.split == "train"])

train RF accuracy: 0.9997
dev RF accuracy: 0.9926
test RF accuracy: 0.9901


,n_flows,rf_class_recall,rf_alert_rate,if_alert_rate,any_alert_rate
family,,,,,
benign,526973,1.000,0.000,0.005,0.005
botnet,1756,1.000,1.000,0.000,1.000
ddos,6198,1.000,1.000,0.000,1.000
dos,160354,1.000,1.000,0.005,1.000
ftp_bruteforce,1703,1.000,1.000,0.000,1.000
other,15,0.733,0.933,0.200,1.000
portscan,181171,0.999,0.999,0.004,1.000
ssh_bruteforce,572,0.997,0.998,0.000,0.998
web_attack,795,0.985,0.986,0.000,0.986


In [4]:
print(len(meta["features"]), "features")
print([c for c in meta["features"] if "ip" in c.split("_") or c in ("timestamp", "flow_id")])
for c, v in meta["top_importances"]:
    print(f"{c:35s} {v:.3f}")

83 features
[]
flow_iat_max                        0.075
flow_duration                       0.071
bwd_segment_size_avg                0.068
packet_length_std                   0.065
packet_length_variance              0.063
packet_length_max                   0.055
bwd_packet_length_std               0.054
flow_bytes_s                        0.036
fwd_packet_length_max               0.035
bwd_packet_length_max               0.031
average_packet_size                 0.031
bwd_packet_length_mean              0.031
packet_length_mean                  0.029
total_length_of_bwd_packet          0.029
total_length_of_fwd_packet          0.028


In [5]:
pd.crosstab(held.family, held.rf_class)

rf_class,benign,botnet,ddos,dos,ftp_bruteforce,other,portscan,ssh_bruteforce,web_attack
family,,,,,,,,,
benign,656835,3,0,127,0,0,59,0,8
botnet,25,1749,0,0,0,0,0,0,0
ddos,21,0,49999,0,0,0,0,0,0
dos,6301,0,0,7739,0,0,0,0,1
ftp_bruteforce,1,0,0,0,1698,0,0,0,0
other,28,15,0,0,0,1,24,0,0
portscan,15,0,0,0,0,0,10665,0,2
ssh_bruteforce,12,0,0,0,0,0,1,1125,0
web_attack,16,0,0,0,0,0,0,0,742


In [6]:
a = held[held.alert]
print(a.groupby(["split"]).size())
print(a.groupby(["family", "split"]).size().unstack(fill_value=0))

split
dev     40891
test    36422
dtype: int64
split             dev   test
family                      
benign           1057   2156
botnet            522   1227
ddos            29381  20618
dos              1550   6512
ftp_bruteforce      1   1698
other              26     29
portscan         7592   3076
ssh_bruteforce    586    540
web_attack        176    566
